creating a book inventory and analysis system


In [ ]:

"""
Bookstore Inventory and Analytics System
Red & White Skill Education
"""

import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


INVENTORY_FILE = "inventory.csv"
SALES_FILE = "sales.csv"

INVENTORY_COLUMNS = ["Title", "Author", "Genre", "Price", "Quantity"]
SALES_COLUMNS = ["Date", "Title", "Quantity Sold", "Total Revenue"]


class Bookstore:

    def __init__(self):
        self.inventory = pd.DataFrame(columns=INVENTORY_COLUMNS)
        self.sales = pd.DataFrame(columns=SALES_COLUMNS)
        self.load_data()

    # Load CSV files or create sample datasets
    def load_data(self):
        try:
            if os.path.exists(INVENTORY_FILE):
                self.inventory = pd.read_csv(INVENTORY_FILE)
            else:
                self.inventory = pd.DataFrame([
                    ["The Alchemist", "Paulo Coelho", "Fiction", 350, 12],
                    ["Wings of Fire", "A. P. J. Abdul Kalam", "Biography", 250, 15],
                    ["Atomic Habits", "James Clear", "Self-help", 499, 10],
                    ["The Hobbit", "J. R. R. Tolkien", "Fantasy", 450, 8],
                    ["Ikigai", "Hector Garcia", "Self-help", 299, 14],
                    ["Malgudi Days", "R. K. Narayan", "Fiction", 275, 9]
                ], columns=INVENTORY_COLUMNS)

                self.save_inventory()

            if os.path.exists(SALES_FILE):
                self.sales = pd.read_csv(SALES_FILE)
            else:
                self.sales = pd.DataFrame(columns=SALES_COLUMNS)
                self.save_sales()

            self.clean_data()

        except (OSError, ValueError, pd.errors.ParserError) as e:
            print("Error loading files:", e)

    # Clean missing and invalid data
    def clean_data(self):
        for column in INVENTORY_COLUMNS:
            if column not in self.inventory.columns:
                self.inventory[column] = np.nan

        self.inventory = self.inventory[INVENTORY_COLUMNS].copy()

        self.inventory["Title"] = (
            self.inventory["Title"].fillna("").astype(str).str.strip()
        )
        self.inventory["Author"] = self.inventory["Author"].fillna("Unknown")
        self.inventory["Genre"] = self.inventory["Genre"].fillna("Other")

        self.inventory["Price"] = pd.to_numeric(
            self.inventory["Price"], errors="coerce"
        )
        self.inventory["Quantity"] = pd.to_numeric(
            self.inventory["Quantity"], errors="coerce"
        )

        self.inventory.dropna(
            subset=["Price", "Quantity"], inplace=True
        )

        self.inventory = self.inventory[
            (self.inventory["Title"] != "") &
            (self.inventory["Price"] >= 0) &
            (self.inventory["Quantity"] >= 0)
        ].copy()

        self.inventory["Quantity"] = (
            self.inventory["Quantity"].astype(int)
        )

        for column in SALES_COLUMNS:
            if column not in self.sales.columns:
                self.sales[column] = np.nan

        self.sales = self.sales[SALES_COLUMNS].copy()

        self.sales["Date"] = pd.to_datetime(
            self.sales["Date"], errors="coerce"
        )
        self.sales["Title"] = (
            self.sales["Title"].fillna("").astype(str).str.strip()
        )
        self.sales["Quantity Sold"] = pd.to_numeric(
            self.sales["Quantity Sold"], errors="coerce"
        )
        self.sales["Total Revenue"] = pd.to_numeric(
            self.sales["Total Revenue"], errors="coerce"
        )

        self.sales.dropna(
            subset=["Date", "Quantity Sold", "Total Revenue"],
            inplace=True
        )

        self.sales = self.sales[
            (self.sales["Title"] != "") &
            (self.sales["Quantity Sold"] > 0) &
            (self.sales["Total Revenue"] >= 0)
        ].copy()

        if not self.sales.empty:
            self.sales["Quantity Sold"] = (
                self.sales["Quantity Sold"].astype(int)
            )

    # Save inventory
    def save_inventory(self):
        try:
            self.inventory.to_csv(INVENTORY_FILE, index=False)
        except OSError as e:
            print("Unable to save inventory:", e)

    # Save sales
    def save_sales(self):
        try:
            data = self.sales.copy()

            if not data.empty:
                data["Date"] = pd.to_datetime(
                    data["Date"], errors="coerce"
                ).dt.strftime("%Y-%m-%d")

            data.to_csv(SALES_FILE, index=False)

        except OSError as e:
            print("Unable to save sales:", e)

    # Add a new book
    def add_book(self, title, author, genre, price, quantity):
        title = title.strip()

        if not title or not author.strip() or not genre.strip():
            print("Title, author and genre cannot be empty.")
            return

        if price < 0 or quantity < 0:
            print("Price and quantity cannot be negative.")
            return

        matching = (
            self.inventory["Title"].str.casefold()
            == title.casefold()
        )

        if matching.any():
            index = self.inventory.index[matching][0]

            self.inventory.loc[index, "Quantity"] += quantity
            self.inventory.loc[index, "Price"] = price
            self.inventory.loc[index, "Author"] = author.strip()
            self.inventory.loc[index, "Genre"] = genre.strip()

            print("Existing book updated and stock increased.")

        else:
            new_book = pd.DataFrame(
                [[title, author.strip(), genre.strip(),
                  price, quantity]],
                columns=INVENTORY_COLUMNS
            )

            self.inventory = pd.concat(
                [self.inventory, new_book],
                ignore_index=True
            )

            print("Book added successfully.")

        self.save_inventory()

    # Remove a book
    def remove_book(self, title):
        matching = (
            self.inventory["Title"].str.casefold()
            == title.strip().casefold()
        )

        if not matching.any():
            print("Book not found.")
            return

        self.inventory = self.inventory.loc[
            ~matching
        ].reset_index(drop=True)

        self.save_inventory()
        print("Book removed successfully.")

    # Update book stock
    def update_inventory(self, title, quantity):
        matching = (
            self.inventory["Title"].str.casefold()
            == title.strip().casefold()
        )

        if not matching.any():
            print("Book not found.")
            return

        if quantity < 0:
            print("Quantity cannot be negative.")
            return

        self.inventory.loc[matching, "Quantity"] = quantity
        self.inventory["Quantity"] = (
            self.inventory["Quantity"].astype(int)
        )

        self.save_inventory()
        print("Stock updated successfully.")

    # Record book sales
    def record_sale(self, title, quantity):
        matching = (
            self.inventory["Title"].str.casefold()
            == title.strip().casefold()
        )

        if not matching.any():
            print("Book not found.")
            return

        if quantity <= 0:
            print("Quantity must be greater than zero.")
            return

        index = self.inventory.index[matching][0]
        available = int(self.inventory.loc[index, "Quantity"])

        if quantity > available:
            print("Insufficient stock. Available:", available)
            return

        price = float(self.inventory.loc[index, "Price"])
        revenue = price * quantity

        sale = pd.DataFrame(
            [[datetime.now().strftime("%Y-%m-%d"),
              self.inventory.loc[index, "Title"],
              quantity, revenue]],
            columns=SALES_COLUMNS
        )

        self.sales = pd.concat(
            [self.sales, sale], ignore_index=True
        )

        self.inventory.loc[index, "Quantity"] = (
            available - quantity
        )

        self.save_inventory()
        self.save_sales()

        print("Sale recorded successfully.")
        print("Total revenue: Rs.", round(revenue, 2))

    # Display inventory
    def show_inventory(self):
        if self.inventory.empty:
            print("Inventory is empty.")
        else:
            print("\nBOOK INVENTORY")
            print(self.inventory.to_string(index=False))

    # Generate inventory and sales report
    def generate_report(self):
        print("\n========== BOOKSTORE REPORT ==========")

        print("Number of book titles:", len(self.inventory))

        total_stock = int(self.inventory["Quantity"].sum())
        print("Total books in stock:", total_stock)

        quantities = self.inventory["Quantity"].to_numpy(dtype=float)
        prices = self.inventory["Price"].to_numpy(dtype=float)

        inventory_value = np.sum(quantities * prices)
        print("Inventory value: Rs.", round(float(inventory_value), 2))

        average_price = np.mean(prices) if len(prices) else 0
        print("Average book price: Rs.", round(float(average_price), 2))

        if self.sales.empty:
            print("No sales recorded yet.")
            return

        print("Total units sold:", int(self.sales["Quantity Sold"].sum()))

        total_revenue = self.sales["Total Revenue"].sum()
        print("Total revenue: Rs.", round(float(total_revenue), 2))

        book_sales = self.sales.groupby("Title").agg(
            Units_Sold=("Quantity Sold", "sum"),
            Revenue=("Total Revenue", "sum")
        ).sort_values("Units_Sold", ascending=False)

        print("\nSales by book:")
        print(book_sales.to_string())

        print("\nBest-selling book:", book_sales.index[0])

        monthly = self.sales.copy()
        monthly["Month"] = pd.to_datetime(
            monthly["Date"]
        ).dt.to_period("M").astype(str)

        print("\nMonthly revenue:")
        print(
            monthly.groupby("Month")["Total Revenue"].sum().to_string()
        )

    # Import inventory from another CSV file
    def load_external_csv(self):
        path = input("Enter inventory CSV path: ").strip()

        if not path:
            return

        try:
            data = pd.read_csv(path)

            missing = [
                column for column in INVENTORY_COLUMNS
                if column not in data.columns
            ]

            if missing:
                print("Missing columns:", ", ".join(missing))
                return

            self.inventory = data[INVENTORY_COLUMNS].copy()
            self.clean_data()
            self.save_inventory()

            print("Inventory imported successfully.")

        except FileNotFoundError:
            print("File not found. Check the file path.")

        except (
            PermissionError, OSError,
            pd.errors.ParserError, UnicodeDecodeError
        ) as e:
            print("Unable to import file:", e)

    # Visualize sales data
    def visualize(self):
        if self.sales.empty:
            print("Record sales before creating charts.")
            return

        sales = self.sales.copy()
        sales["Date"] = pd.to_datetime(
            sales["Date"], errors="coerce"
        )
        sales.dropna(subset=["Date"], inplace=True)

        if sales.empty:
            print("No valid sales dates found.")
            return

        print("\n1. Bar chart: Units sold by book")
        print("2. Line graph: Monthly revenue")
        print("3. Pie chart: Revenue by genre")
        print("4. Heatmap: Price and units sold")

        choice = input("Choose visualization: ").strip()

        try:
            sns.set_theme(style="whitegrid")

            if choice == "1":
                summary = sales.groupby("Title")[
                    "Quantity Sold"
                ].sum().sort_values(ascending=False)

                summary.plot(
                    kind="bar",
                    figsize=(10, 5),
                    title="Units Sold by Book"
                )

                plt.xlabel("Book Title")
                plt.ylabel("Units Sold")
                plt.xticks(rotation=35, ha="right")

            elif choice == "2":
                sales["Month"] = (
                    sales["Date"].dt.to_period("M").astype(str)
                )

                monthly = sales.groupby("Month")[
                    "Total Revenue"
                ].sum()

                plt.figure(figsize=(9, 5))

                sns.lineplot(
                    x=monthly.index,
                    y=monthly.values,
                    marker="o"
                )

                plt.title("Monthly Sales Revenue")
                plt.xlabel("Month")
                plt.ylabel("Revenue (Rs.)")
                plt.xticks(rotation=35, ha="right")

            elif choice == "3":
                sales_genre = sales.merge(
                    self.inventory[["Title", "Genre"]].drop_duplicates(
                        "Title"
                    ),
                    on="Title",
                    how="left"
                )

                sales_genre["Genre"] = (
                    sales_genre["Genre"].fillna("Unknown")
                )

                genre_revenue = sales_genre.groupby("Genre")[
                    "Total Revenue"
                ].sum()

                if genre_revenue.sum() <= 0:
                    print("No positive revenue available for a pie chart.")
                    return

                plt.figure(figsize=(7, 7))

                plt.pie(
                    genre_revenue,
                    labels=genre_revenue.index,
                    autopct="%1.1f%%",
                    startangle=90
                )

                plt.title("Revenue Share by Genre")
                plt.tight_layout()
                plt.show()
                self.save_chart()
                return

            elif choice == "4":
                book_sales = sales.groupby("Title")[
                    "Quantity Sold"
                ].sum().reset_index()

                price_sales = self.inventory[
                    ["Title", "Price"]
                ].merge(book_sales, on="Title", how="inner")

                if len(price_sales) < 2:
                    print("At least two sold books are needed.")
                    return

                correlation = price_sales[
                    ["Price", "Quantity Sold"]
                ].corr(numeric_only=True)

                plt.figure(figsize=(6, 4))

                sns.heatmap(
                    correlation,
                    annot=True,
                    cmap="coolwarm",
                    vmin=-1,
                    vmax=1
                )

                plt.title("Correlation: Price and Units Sold")
                plt.tight_layout()
                plt.show()
                self.save_chart()
                return

            else:
                print("Invalid chart choice.")
                return

            plt.tight_layout()
            plt.show()
            self.save_chart()

        except (ValueError, OSError) as e:
            print("Unable to create chart:", e)

    # Save visualization as PNG
    def save_chart(self):
        answer = input("Save chart as PNG? (y/n): ").strip().lower()

        if answer == "y":
            filename = input("Enter filename: ").strip()

            if not filename:
                filename = "bookstore_chart.png"

            if not filename.lower().endswith(".png"):
                filename += ".png"

            try:
                plt.savefig(
                    filename,
                    dpi=300,
                    bbox_inches="tight"
                )
                print("Chart saved:", filename)

            except OSError as e:
                print("Unable to save chart:", e)

        plt.close("all")


# Input validation function
def get_number(prompt, number_type=float, minimum=None):
    while True:
        try:
            value = number_type(input(prompt))

            if minimum is not None and value < minimum:
                print("Value must be at least", minimum)
                continue

            return value

        except ValueError:
            print("Invalid input. Enter a valid number.")


# Main menu
def main():
    bookstore = Bookstore()

    while True:
        print("\n===== BOOKSTORE INVENTORY AND ANALYTICS =====")
        print("1. Show inventory")
        print("2. Add a book")
        print("3. Remove a book")
        print("4. Update stock")
        print("5. Record a sale")
        print("6. Generate report")
        print("7. Create visualization")
        print("8. Import inventory CSV")
        print("9. Save all data")
        print("10. Exit")

        choice = input("Enter your choice (1-10): ").strip()

        try:
            if choice == "1":
                bookstore.show_inventory()

            elif choice == "2":
                title = input("Book title: ")
                author = input("Author: ")
                genre = input("Genre: ")

                price = get_number("Price (Rs.): ", float, 0)
                quantity = get_number("Quantity: ", int, 0)

                bookstore.add_book(
                    title, author, genre, price, quantity
                )

            elif choice == "3":
                title = input("Title to remove: ")
                bookstore.remove_book(title)

            elif choice == "4":
                title = input("Book title: ")
                quantity = get_number(
                    "New stock quantity: ", int, 0
                )
                bookstore.update_inventory(title, quantity)

            elif choice == "5":
                title = input("Book title sold: ")
                quantity = get_number(
                    "Quantity sold: ", int, 1
                )
                bookstore.record_sale(title, quantity)

            elif choice == "6":
                bookstore.generate_report()

            elif choice == "7":
                bookstore.visualize()

            elif choice == "8":
                bookstore.load_external_csv()

            elif choice == "9":
                bookstore.save_inventory()
                bookstore.save_sales()
                print("All data saved.")

            elif choice == "10":
                bookstore.save_inventory()
                bookstore.save_sales()
                print("Thank you for using the system!")
                break

            else:
                print("Choose a number from 1 to 10.")

        except (KeyboardInterrupt, EOFError):
            print("\nInput cancelled. Returning to menu.")

        except Exception as e:
            print("An unexpected error occurred:", e)


if __name__ == "__main__":
    main()
